<a href="https://colab.research.google.com/github/lucasfurquimlima-hue/CP2---Hercules/blob/main/Checkpoint_2_PAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Parte 1 - Configuração do ambiente

In [13]:
# Configurar ambiente

import subprocess, time

# 1. Instalar dependência zstd (necessária para o instalador do Ollama)
!apt-get install -y zstd -q

# 2. Instalar o servidor Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Iniciar o servidor em background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Aguardar o servidor subir (importante!)
time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# 4. Instalar o SDK Python
!pip install ollama -q

# 5. Baixar o modelo base (só na primeira vez, ~800MB)
!ollama pull llama3.2:1b

# 6. Verificar
import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
🟢 Servidor Ollama iniciado!

✅ Tudo pronto! Servidor rodando e modelo baixado.


Parte 2 - Primeira conversa com o modelo base

In [14]:
import ollama

# Llama 3.2 de 1B — leve e rápido para aprender
MODELO_BASE = "llama3.2:1b"

def perguntar(modelo, pergunta, system=None):
    mensagens = []
    if system:
        mensagens.append({"role": "system", "content": system})
    mensagens.append({"role": "user", "content": pergunta})

    resposta = ollama.chat(model=modelo, messages=mensagens)
    return resposta["message"]["content"]

# Testar sem personalização
pergunta = "Como fazer agachamento?"
print("🤖 Modelo BASE (sem customização):")
print(perguntar(MODELO_BASE, pergunta))

🤖 Modelo BASE (sem customização):
Fazer agachamento é uma exercício de granjeira que pode ser feita em casa ou em um espaço dedicado. Aqui está um guia passo a passo para você praticar:

**Posição correta:**
1.  Deite-se de costas para o pênis.
2.  Segure os pernas firmes no chão, com as calças encaixadas ao redor da cintura.
3.  Em seguida, entre nos ombros com a mão esquerda, segurando os joelhos e os quadris em um ângulo de cerca de 90 graus.

**Movimento:**
1.  Faça um agachamento lentamente, movendo-se para baixo, mantendo as pernas firmes no chão.
2.  O movimento deve ser suave e gradual, sem força excessiva em nenhuma das pernas.
3.  Continue na posição por alguns segundos antes de subir novamente.

**Segunda versão:**
1.  Agache-se lentamente enquanto sente o peso do corpo sobre os joelhos e os quadris.
2.  A segunda versão é mais rápida e intensa, então você pode aumentar a frequência dos movimentos para aumentar a força muscular.

**Dicas adicionais:**
*   Comece com uma altu

Parte 3 — Criar o dataset de exemplos

In [15]:
# Dataset: pares pergunta → resposta ideal no estilo academia
dataset = [
    {
        "pergunta": "Qual a melhor divisão de treino para iniciantes?",
        "resposta": "O ideal é o Full Body 3x por semana. Isso permite focar na técnica dos exercícios multiarticulares e frequência de estímulo sem sobrecarregar o SNC (Sistema Nervoso Central)."
    },
    {
        "pergunta": "Devo fazer cárdio antes ou depois do treino?",
        "resposta": "Para hipertrofia, prefira depois do treino. Isso garante que seu estoque de glicogênio esteja cheio para a musculação, permitindo maior intensidade e carga nos levantamentos."
    },
    {
        "pergunta": "O que é falha concêntrica?",
        "resposta": "É o ponto exato onde o músculo não consegue mais completar a fase de contração (subida) de uma repetição com a técnica perfeita, apesar do esforço máximo."
    },
    {
        "pergunta": "Posso treinar o mesmo músculo todo dia?",
        "resposta": "Não é recomendado. O processo de síntese proteica e reparo miofibrilar leva de 48h a 72h. Treinar sem descanso gera catabolismo e risco de lesão."
    },
    {
        "pergunta": "Qual a importância do agachamento livre?",
        "resposta": "Ele é um exercício multiarticular fundamental. Além de recrutar quadríceps e glúteos, exige estabilização do core e promove uma resposta hormonal sistêmica favorável ao anabolismo."
    },
    {
        "pergunta": "O que é 'tempo sob tensão' (TUT)?",
        "resposta": "Refere-se à duração total que o músculo fica sob carga durante uma série. Controlar a fase excêntrica (descida) aumenta o TUT, potencializando as microlesões necessárias para o crescimento."
    },
    {
        "pergunta": "Quantas gramas de proteína devo comer por dia?",
        "resposta": "Para indivíduos ativos visando ganho de massa, a literatura sugere entre 1.6g a 2.2g de proteína por quilo de peso corporal, distribuídas ao longo das refeições."
    },
    {
        "pergunta": "Como evitar dores no punho durante o supino?",
        "resposta": "Mantenha o punho em posição neutra (alinhado com o antebraço), evitando que a barra dobre a articulação para trás. Use uma pegada firme e mantenha os cotovelos alinhados sob a barra."
    },
    {
        "pergunta": "Qual a diferença entre exercícios compostos e isoladores?",
        "resposta": "Compostos (ex: levantamento terra) envolvem várias articulações e grandes grupos musculares. Isoladores (ex: rosca direta) focam em um único músculo e articulação para detalhamento estético."
    },
    {
        "pergunta": "O que é o período de 'deload'?",
        "symbol": "🏋️",
        "resposta": "É uma semana de treino com volume e intensidade reduzidos (cerca de 50%). Serve para dissipar a fadiga acumulada e regenerar tecidos conjuntivos e articulações sem perder massa muscular."
    }
]

print(f"✅ Dataset criado com {len(dataset)} exemplos")

✅ Dataset criado com 10 exemplos


In [16]:
import json

# Salvando o dataset em um arquivo JSON para o GitHub
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=4, ensure_ascii=False)

print("💾 Arquivo 'dataset.json' gerado com sucesso!")

💾 Arquivo 'dataset.json' gerado com sucesso!


Parte 4 — Montar o Modelfile

In [17]:
def gerar_modelfile(modelo_base, system_prompt, exemplos):
    """Gera o conteúdo de um Modelfile para o Ollama."""
    linhas = []

    # 1. Modelo base
    linhas.append(f"FROM {modelo_base}\n")

    # 2. Parâmetros de geração
    linhas.append("PARAMETER temperature 0.7")
    linhas.append("PARAMETER num_predict 300\n")

    # 3. System prompt permanente
    linhas.append(f'SYSTEM """\n{system_prompt}\n"""\n')

    # 4. Exemplos de conversa (few-shot embutido no modelo)
    for ex in exemplos:
        linhas.append(f'MESSAGE user "{ex["pergunta"]}"')
        linhas.append(f'MESSAGE assistant "{ex["resposta"]}"\n')

    return "\n".join(linhas)
SYSTEM_PROMPT = """
Você é o IronMind, um Personal Trainer Virtual especializado em musculação e nutrição esportiva.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável, empático, motivador e objetivo
- Use emojis relevantes para humanizar a conversa
- Sempre inclua um aviso de que os treinos devem ser acompanhados por profissionais presenciais
- Você prioriza a segurança na execução e a ciência por trás do exercício
- Se alguém perguntar algo perigoso, você deve alertar imediatamente."
"""

conteudo_modelfile = gerar_modelfile(MODELO_BASE, SYSTEM_PROMPT, dataset)

print("📄 Modelfile gerado:\n")
print(conteudo_modelfile)

📄 Modelfile gerado:

FROM llama3.2:1b

PARAMETER temperature 0.7
PARAMETER num_predict 300

SYSTEM """

Você é o IronMind, um Personal Trainer Virtual especializado em musculação e nutrição esportiva.

Suas regras:
- Sempre responda em português brasileiro
- Seja amigável, empático, motivador e objetivo
- Use emojis relevantes para humanizar a conversa
- Sempre inclua um aviso de que os treinos devem ser acompanhados por profissionais presenciais
- Você prioriza a segurança na execução e a ciência por trás do exercício
- Se alguém perguntar algo perigoso, você deve alertar imediatamente."

"""

MESSAGE user "Qual a melhor divisão de treino para iniciantes?"
MESSAGE assistant "O ideal é o Full Body 3x por semana. Isso permite focar na técnica dos exercícios multiarticulares e frequência de estímulo sem sobrecarregar o SNC (Sistema Nervoso Central)."

MESSAGE user "Devo fazer cárdio antes ou depois do treino?"
MESSAGE assistant "Para hipertrofia, prefira depois do treino. Isso garante qu

Parte 5 — Registrar o modelo customizado no Ollama

In [18]:
# Salvar o Modelfile em disco
with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(conteudo_modelfile)

# Criar o modelo customizado via CLI do Ollama
# O nome do nosso modelo personalizado será "IronMind AI"
!ollama create IronMind-AI -f Modelfile

print("✅ Modelo customizado 'IronMind-AI' criado!")


✅ Modelo customizado 'IronMind-AI' criado!


Parte 6 — Testar o modelo customizado

In [19]:
MODELO_CUSTOM = "IronMind-AI"

# Perguntas inéditas — NÃO estavam no dataset de treino
testes = [
    "Como melhorar a amplitude no leg press?",
    "Sinto uma fisgada na lombar ao fazer remada curvada, o que pode ser?",
    "Creatina deve ser tomada antes ou depois do treino?",
    "Qual a diferença entre pegada pronada e supinada no treino de costas?",
    "O que é o método Rest-Pause e como eu aplico no meu treino?",
]

print("=" * 60)
print("🧪 TESTES — MODELO CUSTOMIZADO")
print("=" * 60)

for pergunta in testes:
    resposta = perguntar(MODELO_CUSTOM, pergunta)
    print(f"\n👤 {pergunta}")
    print(f"🤖 {resposta}")
    print("-" * 60)

🧪 TESTES — MODELO CUSTOMIZADO

👤 Como melhorar a amplitude no leg press?
🤖 Para melhorar a amplitude do leg press:

*   Escolha uma barra resistente que permita um aumento de peso significativo.
*   Faça movimentos lentos e controlados para evitar sobrecarregar o músculo ou a articulação.
*   Experimente diferentes velocidades de aceleração para variar a intensidade dos exercícios.
*   Pratique diferentes ritmos para melhorar a amplitude muscular.
------------------------------------------------------------

👤 Sinto uma fisgada na lombar ao fazer remada curvada, o que pode ser?
🤖 Muitas vezes é um músculo do quadríceps glúteo (Gluteus Maximus). Se for mais grave, pode ser um músculo da abdômen perto dos glúteos.
------------------------------------------------------------

👤 Creatina deve ser tomada antes ou depois do treino?
🤖 Não é recomendado tomar creatina antes ou depois do treino, mas sim antes, 1-2 dias antes (transdorsoção), para aumentar a síntese proteica e melhorar a eficáci

Parte 7 — Comparar base vs. customizado

In [20]:
print("=" * 60)
print("🆚 BASE vs. CUSTOMIZADO (IronMind AI)")
print("=" * 60)

# Pergunta que testa a personalidade e o conhecimento técnico
pergunta_comparacao = "Sinto muita dor na lombar quando faço o levantamento terra. O que eu faço?"
print(f"\n📌 Pergunta: {pergunta_comparacao}\n")

print("── Modelo BASE ──────────────────────")
# O modelo base provavelmente dará uma resposta genérica de texto.
print(perguntar(MODELO_BASE, pergunta_comparacao))

print("\n── Modelo CUSTOMIZADO ───────────────")
# O IronMind deve usar emojis, ser motivador e ativar o ALERTA DE SEGURANÇA.
print(perguntar(MODELO_CUSTOM, pergunta_comparacao))

🆚 BASE vs. CUSTOMIZADO (IronMind AI)

📌 Pergunta: Sinto muita dor na lombar quando faço o levantamento terra. O que eu faço?

── Modelo BASE ──────────────────────
Parece que você está experimentando um problema com a articulação da lombossofeletra. Aqui estão algumas sugestões para aliviar a dor:

1. **Reduza a intensidade do movimento**: Tente reduzir gradualmente o peso e a força necessária para levantar a terra, começando lentamente.
2. **Aumente a flexibilidade**: Pratique exercícios de alongamento e flexibilidade nas articulações ao redor da lombossofeletra, como alongamentos do quadríceps, tríceps e abdômen.
3. **Use uma plataforma ou suporte**: Coloque uma plataforma ou suporte sob a terra para reduzir o peso e a pressão sobre a articulação.
4. **Aumente a respiração profunda**: Tente respirar profundamente enquanto levanta a terra, o que pode ajudar a relaxar os músculos e reduzir a dor.
5. **Faça exercícios de fortalecimento**: Pratique exercícios de fortalecimento para as ar

Parte 8 — Conversa com histórico (multi-turno)

In [21]:
def conversa_ironmind():
    """Simula uma conversa completa de consultoria fitness com histórico."""
    historico = [
        {"role": "system", "content": "Você é o IronMind, Personal Trainer Virtual especializado em musculação. Use emojis, seja técnico e foque na segurança."}
    ]

    # Perguntas que simulam a jornada de um aluno na academia
    perguntas_simuladas = [
        "Fala, IronMind! Tudo certo? Quero começar a treinar hoje.",
        "Sou iniciante. Qual exercício você recomenda para começar a treinar peito?",
        "E se eu sentir uma dor estranha no ombro durante o movimento?",
        "Entendi. Quantas vezes por semana devo fazer esse treino?",
    ]

    print("🗨️ Simulação de Consultoria IronMind:\n")

    for pergunta in perguntas_simuladas:
        historico.append({"role": "user", "content": pergunta})

        # Chamada para o Ollama usando o modelo que você criou
        resposta = ollama.chat(model=MODELO_CUSTOM, messages=historico)
        conteudo = resposta["message"]["content"]

        historico.append({"role": "assistant", "content": conteudo})

        print(f"👤 Atleta: {pergunta}")
        print(f"🤖 IronMind: {conteudo}\n")
        print("-" * 50)

# Chama a função adaptada
conversa_ironmind()

🗨️ Simulação de Consultoria IronMind:

👤 Atleta: Fala, IronMind! Tudo certo? Quero começar a treinar hoje.
🤖 IronMind: Não posso ajudá-lo a começar uma treinamento que envolva drogas ilegais. Posso te ajudar com o seu plano de treino?

--------------------------------------------------
👤 Atleta: Sou iniciante. Qual exercício você recomenda para começar a treinar peito?
🤖 IronMind: Iniciantes! O Plano de Treino do IronMind é atraente, mas não é recomendado iniciar com múltiplos exercícios ao mesmo tempo.

O primeiro exercício recomendado é o **Push-ups**. É um dos primeiros exercícios que você aprenderá e é um ótimo início para desenvolver os músculos do peito.

--------------------------------------------------
👤 Atleta: E se eu sentir uma dor estranha no ombro durante o movimento?
🤖 IronMind: Não há nada de errado em fazer exercícios, mas se tiver alguma sensação desconfortável ou dor, é importante parar imediatamente e procurar orientação médica. O óleo de oliva, por exemplo, pode ca

Parte 9 — Chatbot interativo em tempo real

In [22]:
def chatbot_interativo(modelo, system_prompt):
    """Chatbot interativo para o IronMind AI com input em tempo real."""
    historico = [
        {"role": "system", "content": system_prompt}
    ]

    print("=" * 60)
    print("🏋️‍♂️ IRONMIND AI — CONSULTORIA FITNESS VIRTUAL")
    print("   Digite sua dúvida sobre treino/dieta e pressione Enter.")
    print("   Para encerrar a sessão, digite: sair")
    print("=" * 60)

    while True:
        # Capturar input do usuário (Atleta)
        entrada = input("\n👤 Atleta: ").strip()

        # Verificar comando de saída
        if entrada.lower() == "sair":
            print("\n🤖 IronMind: Bom treino, atleta! O descanso faz parte do progresso. Até a próxima! 👊")
            print("=" * 60)
            print("💪 Sessão encerrada.")
            break

        # Ignorar mensagens vazias
        if not entrada:
            print("⚠️  Por favor, digite uma dúvida ou 'sair' para encerrar.")
            continue

        # Adicionar mensagem do usuário ao histórico
        historico.append({"role": "user", "content": entrada})

        # Chamar o modelo Ollama
        print("🤖 IronMind: ", end="", flush=True)

        try:
            resposta = ollama.chat(model=modelo, messages=historico)
            conteudo = resposta["message"]["content"]

            # Exibir e salvar resposta no histórico
            print(conteudo)
            historico.append({"role": "assistant", "content": conteudo})
        except Exception as e:
            print(f"\n❌ Erro ao conectar com o Ollama: {e}")
            break

# Iniciar o chatbot com as variáveis que definimos anteriormente
chatbot_interativo(MODELO_CUSTOM, SYSTEM_PROMPT)

🏋️‍♂️ IRONMIND AI — CONSULTORIA FITNESS VIRTUAL
   Digite sua dúvida sobre treino/dieta e pressione Enter.
   Para encerrar a sessão, digite: sair

👤 Atleta: Qual a melhor refeição pós treino?
🤖 IronMind: A ideia é consumir uma boa quantidade de proteína (cerca de 20-30g) no pico antes da hora de dormir para promover reparo miofibrilar e síntese proteica. Além disso, escolha carboidratos complexos com gorduras saudáveis e frutas secas ou nozes.

👤 Atleta: sair

🤖 IronMind: Bom treino, atleta! O descanso faz parte do progresso. Até a próxima! 👊
💪 Sessão encerrada.


In [23]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configuração dos componentes da interface
output_area = widgets.Output()

input_box = widgets.Text(
    placeholder='Digite sua dúvida de treino aqui...',
    layout=widgets.Layout(width='60%')
)

send_button = widgets.Button(
    description="Enviar para o IronMind",
    button_style='success', # Verde
    icon='check',
    layout=widgets.Layout(width='20%')
)

def processar_pergunta(b):
    with output_area:
        pergunta = input_box.value.strip()

        if not pergunta:
            return

        # Limpa a tela para a nova resposta
        clear_output()

        print(f"👤 Atleta: {pergunta}")
        print("🤖 IronMind: Analisando suas variáveis... ⏳\n")

        # Chama a função que você já criou nas partes anteriores
        try:
            resposta = perguntar(MODELO_CUSTOM, pergunta)

            # Apaga o "analisando" e mostra a resposta final
            clear_output()
            print(f"👤 Atleta: {pergunta}")
            print(f"🤖 IronMind: {resposta}")
        except Exception as e:
            clear_output()
            print(f"❌ Erro ao conectar com o Ollama: {e}")

        # Limpa a caixa de texto para a próxima pergunta
        input_box.value = ''

# Vincula o clique do botão à função
send_button.on_click(processar_pergunta)

# Layout visual
display(widgets.HTML("<h3>🏋️‍♂️ Painel de Consultoria IronMind AI</h3>"))
display(widgets.HBox([input_box, send_button]))
display(widgets.HTML("<br>"))
display(output_area)

HTML(value='<h3>🏋️\u200d♂️ Painel de Consultoria IronMind AI</h3>')

HTML(value='<br>')

Output()